# BIST-100 Güniçi Scalp Trading: TA + GARCH + XGBoost Şampiyon Strateji Motoru

**Mimari:** Özellik Mühendisliği → Üç Bağımsız Strateji Motoru → Vektörel Backtest → Kıyaslama & Şampiyon İlanı

| Katman | Açıklama |
|--------|----------|
| Veri | yfinance ile 1d/1h/15m OHLCV + XU100 endeks verisi |
| Strateji A | RSI-14 + MACD sinyal kesişimi (kural tabanlı TA) |
| Strateji B | GARCH(1,1) koşullu oynaklık eşik filtresi |
| Strateji C | XGBoost sınıflandırıcı (teknik + göreceli özellikler) |
| Backtest | Kronolojik %20 test seti, vektörel hesaplama |
| Metrikler | Getiri, Sharpe, MaxDD, Win Rate, Calmar |

In [ ]:
# Gerekli kütüphaneleri kur
import subprocess, sys
pkgs = ['yfinance', 'arch', 'xgboost', 'ta', 'matplotlib', 'seaborn', 'scikit-learn']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=False)
print('Kurulum tamamlandı.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime, timedelta

# İstatistik & ML
from arch import arch_model
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Teknik analiz
import ta

pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('dark_background')
COLORS = ['#00ff88', '#ff6b6b', '#4ecdc4', '#ffd93d', '#c77dff']

print('Tüm kütüphaneler yüklendi ✓')

## ⚙️ KONFIGÜRASYON — Hisseyi, Zaman Dilimini ve Parametreleri Buradan Ayarla

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  KULLANICI AYARLARI  — sadece bu bloğu değiştir
# ═══════════════════════════════════════════════════════════════

TICKER      = 'THYAO.IS'      # BIST hissesi (örn: 'SISE.IS', 'EREGL.IS', 'AKBNK.IS')
INDEX_TKR   = 'XU100.IS'      # Karşılaştırma endeksi
INTERVAL    = '1h'            # '1d' / '1h' / '15m' / '5m'
PERIOD      = '2y'            # '1y' / '2y' / '5y'  (1h için max 2y, 15m için max 60d)

# Teknik parametreler
RSI_PERIOD   = 14
MACD_FAST    = 12
MACD_SLOW    = 26
MACD_SIGNAL  = 9
VOL_MA_WIN   = 20

# GARCH oynaklık eşiği (yüzdelik dilim)
GARCH_THRESH_LOW  = 30   # Oynaklık bu yüzdelimin altında → düşük → potansiyel kırılım
GARCH_THRESH_HIGH = 70   # Oynaklık bu yüzdelimin üstünde → yüksek → kaçın

# XGBoost
TRAIN_RATIO  = 0.80       # Kronolojik bölme oranı
XGB_PARAMS   = dict(
    n_estimators=400, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)

# Backtest
TRANSACTION_COST = 0.001  # İşlem başı maliyet (binde 1 gidiş + binde 1 dönüş)
INITIAL_CAPITAL  = 100_000  # TL

print(f'Konfigürasyon: {TICKER} | {INTERVAL} | {PERIOD}')

## 📥 1. VERİ İNDİRME

In [ ]:
def download_data(ticker: str, index_ticker: str, interval: str, period: str) -> pd.DataFrame:
    """yfinance'dan hisse + endeks verisini indir, birleştir ve temizle."""
    print(f'▶ {ticker} indiriliyor...')
    raw  = yf.download(ticker,       period=period, interval=interval, progress=False, auto_adjust=True)
    idx  = yf.download(index_ticker, period=period, interval=interval, progress=False, auto_adjust=True)

    if raw.empty:
        raise ValueError(f'{ticker} için veri bulunamadı. Hisse kodunu kontrol et.')

    # MultiIndex sütun varsa düzleştir
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    if isinstance(idx.columns, pd.MultiIndex):
        idx.columns = idx.columns.get_level_values(0)

    df = raw[['Open','High','Low','Close','Volume']].copy()
    df.index = pd.to_datetime(df.index)
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    # Endeks kapanışını ekle
    if not idx.empty:
        if idx.index.tz is not None:
            idx.index = idx.index.tz_localize(None)
        df['Endeks_Close'] = idx['Close'].reindex(df.index, method='ffill')
    else:
        df['Endeks_Close'] = df['Close']  # Endeks yoksa kendi fiyatıyla doldur

    df.dropna(subset=['Close'], inplace=True)
    df.sort_index(inplace=True)

    print(f'✓ {len(df):,} bar yüklendi | {df.index[0].date()} → {df.index[-1].date()}')
    return df


df_raw = download_data(TICKER, INDEX_TKR, INTERVAL, PERIOD)
df_raw.tail()

## 🔧 2. ÖZELLİK MÜHENDİSLİĞİ & HEDEF TANIMLAMA

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tüm teknik ve istatistiksel özellikleri hesapla."""
    d = df.copy()

    # ── Temel getiri ──────────────────────────────────────────────
    d['Log_Return']  = np.log(d['Close'] / d['Close'].shift(1))
    d['Next_Return'] = d['Log_Return'].shift(-1)   # hedef için
    d['Label']       = (d['Next_Return'] > 0).astype(int)

    # ── RSI ───────────────────────────────────────────────────────
    d['RSI'] = ta.momentum.RSIIndicator(close=d['Close'], window=RSI_PERIOD).rsi()

    # ── MACD ──────────────────────────────────────────────────────
    macd_obj = ta.trend.MACD(
        close=d['Close'],
        window_slow=MACD_SLOW,
        window_fast=MACD_FAST,
        window_sign=MACD_SIGNAL
    )
    d['MACD']        = macd_obj.macd()
    d['MACD_Signal'] = macd_obj.macd_signal()
    d['MACD_Hist']   = macd_obj.macd_diff()

    # ── Hareketli Ortalamalar ─────────────────────────────────────
    d['EMA9']  = ta.trend.EMAIndicator(d['Close'], window=9).ema_indicator()
    d['EMA21'] = ta.trend.EMAIndicator(d['Close'], window=21).ema_indicator()
    d['SMA50'] = ta.trend.SMAIndicator(d['Close'], window=50).sma_indicator()
    d['SMA200']= ta.trend.SMAIndicator(d['Close'], window=200).sma_indicator()

    # ── Bollinger Bantları ────────────────────────────────────────
    bb = ta.volatility.BollingerBands(d['Close'], window=20, window_dev=2)
    d['BB_High'] = bb.bollinger_hband()
    d['BB_Low']  = bb.bollinger_lband()
    d['BB_Width']= bb.bollinger_wband()
    d['BB_Pct']  = bb.bollinger_pband()   # 0–1 arası pozisyon

    # ── ATR (Average True Range) ──────────────────────────────────
    d['ATR'] = ta.volatility.AverageTrueRange(
        d['High'], d['Low'], d['Close'], window=14
    ).average_true_range()
    d['ATR_Pct'] = d['ATR'] / d['Close']  # normalize

    # ── Stokastik ─────────────────────────────────────────────────
    stoch = ta.momentum.StochasticOscillator(d['High'], d['Low'], d['Close'], window=14, smooth_window=3)
    d['Stoch_K'] = stoch.stoch()
    d['Stoch_D'] = stoch.stoch_signal()

    # ── Hacim Özellikleri ─────────────────────────────────────────
    d['Vol_MA']     = d['Volume'].rolling(VOL_MA_WIN).mean()
    d['Vol_Ratio']  = d['Volume'] / d['Vol_MA']        # hacim baskısı
    d['OBV']        = ta.volume.OnBalanceVolumeIndicator(d['Close'], d['Volume']).on_balance_volume()
    d['OBV_EMA']    = d['OBV'].ewm(span=20).mean()
    d['OBV_Signal'] = (d['OBV'] > d['OBV_EMA']).astype(int)

    # ── Endeks Göreceli Gücü ──────────────────────────────────────
    d['Relative_Strength'] = d['Close'] / d['Endeks_Close']
    d['RS_Momentum']       = d['Relative_Strength'].pct_change(5)  # 5 bar RS değişimi

    # ── Momentum / ROC ────────────────────────────────────────────
    d['ROC5']  = ta.momentum.ROCIndicator(d['Close'], window=5).roc()
    d['ROC10'] = ta.momentum.ROCIndicator(d['Close'], window=10).roc()
    d['ROC20'] = ta.momentum.ROCIndicator(d['Close'], window=20).roc()

    # ── Trend Gücü ────────────────────────────────────────────────
    adx = ta.trend.ADXIndicator(d['High'], d['Low'], d['Close'], window=14)
    d['ADX']   = adx.adx()
    d['DI_pos']= adx.adx_pos()
    d['DI_neg']= adx.adx_neg()

    # ── Fiyat Pattern Özellikleri ─────────────────────────────────
    d['Body_Size']   = abs(d['Close'] - d['Open']) / (d['High'] - d['Low'] + 1e-9)
    d['Upper_Wick']  = (d['High'] - d[['Close','Open']].max(axis=1)) / (d['High'] - d['Low'] + 1e-9)
    d['Lower_Wick']  = (d[['Close','Open']].min(axis=1) - d['Low']) / (d['High'] - d['Low'] + 1e-9)
    d['Bullish_Bar'] = (d['Close'] > d['Open']).astype(int)

    # ── Gecikme (Lag) Özellikleri ─────────────────────────────────
    for lag in [1, 2, 3, 5]:
        d[f'Ret_lag{lag}'] = d['Log_Return'].shift(lag)
        d[f'RSI_lag{lag}'] = d['RSI'].shift(lag)

    # ── Güniçi Volatilite ─────────────────────────────────────────
    d['Intraday_Range'] = (d['High'] - d['Low']) / d['Open']
    d['Gap_Pct']        = (d['Open'] - d['Close'].shift(1)) / d['Close'].shift(1)

    return d


df = build_features(df_raw)
print(f'Özellik sayısı: {df.shape[1]} | Satır: {df.shape[0]:,}')
df[['Close','RSI','MACD','MACD_Signal','ATR_Pct','Vol_Ratio','Label']].tail(10)

## 🏗️ 3. STRATEJİ MOTORLARI

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STRATEJİ A — Klasik Teknik Analiz (RSI + MACD + EMA Filtresi)
# ══════════════════════════════════════════════════════════════════

def strategy_A_signals(df: pd.DataFrame) -> pd.Series:
    """
    Al sinyali  (1) : RSI < 70 VE RSI > 30 VE MACD histogram pozitife döndü
                       VE fiyat EMA9 > EMA21 (kısa vadeli trend yukarı)
    Sat sinyali (-1): RSI > 70 VEYA MACD histogram negatife döndü
                       VEYA fiyat EMA9 < EMA21
    """
    hist         = df['MACD_Hist']
    hist_cross_up = (hist > 0) & (hist.shift(1) <= 0)   # histogram sıfırı yukarı kesiyor
    hist_cross_dn = (hist < 0) & (hist.shift(1) >= 0)

    rsi_ok    = (df['RSI'] > 30) & (df['RSI'] < 70)
    trend_up  = df['EMA9'] > df['EMA21']
    trend_dn  = df['EMA9'] < df['EMA21']

    # Güçlendirici filtreler
    vol_confirm   = df['Vol_Ratio'] > 1.0      # hacim ortalamanın üzerinde
    adx_trending  = df['ADX'] > 20             # yeterli trend gücü

    sig = pd.Series(0, index=df.index)
    sig[hist_cross_up & rsi_ok & trend_up & vol_confirm] = 1
    sig[hist_cross_dn | trend_dn]                        = -1

    return sig


print('Strateji A (Klasik TA) fonksiyonu tanımlandı ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STRATEJİ B — GARCH(1,1) Oynaklık Filtresi
# ══════════════════════════════════════════════════════════════════

def strategy_B_signals(df: pd.DataFrame,
                        thresh_low: float = GARCH_THRESH_LOW,
                        thresh_high: float = GARCH_THRESH_HIGH) -> pd.Series:
    """
    GARCH(1,1) ile koşullu oynaklık (σ_t) hesapla.
    - Oynaklık percentile DÜŞÜK bölgedeyse (sıkışma / fırlatma öncesi):
        RSI yönüne göre al (>50) veya sat (<50)
    - Oynaklık percentile YÜKSEK bölgedeyse:
        Piyasadan çık (0) — yüksek riskten kaçın
    """
    returns = df['Log_Return'].dropna() * 100  # % cinsinden

    # GARCH modelini fit et
    try:
        model  = arch_model(returns, vol='Garch', p=1, q=1, dist='Normal', rescale=False)
        result = model.fit(disp='off', options={'maxiter': 500})
        cond_vol = result.conditional_volatility  # % cinsinden
    except Exception as e:
        print(f'GARCH fit hatası: {e}. Sabit oynaklık kullanılıyor.')
        cond_vol = pd.Series(returns.rolling(20).std().values, index=returns.index)

    # Yüzdelik dilim hesapla (rolling 252 bar)
    vol_pct = cond_vol.rolling(252, min_periods=50).rank(pct=True) * 100
    vol_pct = vol_pct.reindex(df.index)  # df ile hizala

    low_vol  = vol_pct < thresh_low
    high_vol = vol_pct > thresh_high

    # Yön için RSI ve MACD histogram kullan
    bullish = (df['RSI'] > 50) & (df['MACD_Hist'] > 0) & (df['EMA9'] > df['EMA21'])
    bearish = (df['RSI'] < 50) & (df['MACD_Hist'] < 0)

    sig = pd.Series(0, index=df.index)
    sig[low_vol & bullish]  =  1   # sıkışma → yukarı kırılım beklentisi
    sig[low_vol & bearish]  = -1   # sıkışma → aşağı kırılım beklentisi
    sig[high_vol]           =  0   # yüksek oynaklık → bekle

    # GARCH oynaklık serisini de döndür (görselleştirme için)
    df['GARCH_Vol']     = cond_vol.reindex(df.index)
    df['GARCH_Vol_Pct'] = vol_pct

    return sig


print('Strateji B (GARCH) fonksiyonu tanımlandı ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STRATEJİ C — XGBoost Sınıflandırıcı
# ══════════════════════════════════════════════════════════════════

FEATURE_COLS = [
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'EMA9', 'EMA21', 'BB_Pct', 'BB_Width',
    'ATR_Pct', 'Stoch_K', 'Stoch_D',
    'Vol_Ratio', 'OBV_Signal',
    'Relative_Strength', 'RS_Momentum',
    'ROC5', 'ROC10', 'ROC20',
    'ADX', 'DI_pos', 'DI_neg',
    'Body_Size', 'Upper_Wick', 'Lower_Wick', 'Bullish_Bar',
    'Ret_lag1', 'Ret_lag2', 'Ret_lag3', 'Ret_lag5',
    'RSI_lag1', 'RSI_lag2', 'RSI_lag3',
    'Intraday_Range', 'Gap_Pct',
]

def strategy_C_train_predict(df: pd.DataFrame, train_ratio: float = TRAIN_RATIO):
    """
    Kronolojik bölme: train_ratio kadarı eğitim, kalanı test.
    Döndürür: (sinyaller, test_start_idx, model, scaler, feature_importance)
    """
    avail_cols = [c for c in FEATURE_COLS if c in df.columns]
    dfc = df[avail_cols + ['Label']].dropna().copy()

    split_n    = int(len(dfc) * train_ratio)
    train_data = dfc.iloc[:split_n]
    test_data  = dfc.iloc[split_n:]

    X_train, y_train = train_data[avail_cols], train_data['Label']
    X_test,  y_test  = test_data[avail_cols],  test_data['Label']

    scaler  = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # Sınıf dengesi için ağırlık
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = XGBClassifier(**XGB_PARAMS, scale_pos_weight=pos_weight)
    model.fit(
        X_train_sc, y_train,
        eval_set=[(X_test_sc, y_test)],
        verbose=False
    )

    preds_train = model.predict(X_train_sc)
    preds_test  = model.predict(X_test_sc)
    proba_test  = model.predict_proba(X_test_sc)[:, 1]

    train_acc = accuracy_score(y_train, preds_train)
    test_acc  = accuracy_score(y_test,  preds_test)
    print(f'XGBoost | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}')
    print(f'Test dönemi: {test_data.index[0].date()} → {test_data.index[-1].date()} ({len(test_data):,} bar)')

    # Sinyal serisi: test döneminde tahminleri al, geri kalanı 0
    sig = pd.Series(0, index=df.index)
    # 1 = al tahmini, 0 = sat/bekle tahmini → -1 olarak kodla
    sig.loc[test_data.index] = np.where(preds_test == 1, 1, -1)

    feat_imp = pd.Series(model.feature_importances_, index=avail_cols).sort_values(ascending=False)

    return sig, test_data.index[0], model, scaler, feat_imp, test_acc


print('Strateji C (XGBoost) fonksiyonu tanımlandı ✓')

## 🚀 4. STRATEJİLERİ ÇALIŞTIR

In [ ]:
print('═'*60)
print('STRATEJİLER HESAPLANIYOR...')
print('═'*60)

# Strateji A
print('\n[A] Klasik TA sinyalleri hesaplanıyor...')
sig_A = strategy_A_signals(df)
print(f'    Sinyal dağılımı: Al={( sig_A==1).sum()} | Sat={(sig_A==-1).sum()} | Bekle={(sig_A==0).sum()}')

# Strateji B
print('\n[B] GARCH(1,1) fit ediliyor...')
sig_B = strategy_B_signals(df)
print(f'    Sinyal dağılımı: Al={( sig_B==1).sum()} | Sat={(sig_B==-1).sum()} | Bekle={(sig_B==0).sum()}')

# Strateji C
print('\n[C] XGBoost eğitiliyor...')
sig_C, test_start, xgb_model, xgb_scaler, feat_imp, xgb_test_acc = strategy_C_train_predict(df)

print(f'\n✓ Tüm stratejiler hazır. Test dönemi başlangıcı: {test_start}')

## 📊 5. BACKTEST & KIYASLAMA MOTORU

In [ ]:
def run_backtest(df: pd.DataFrame, signals: pd.Series, strategy_name: str,
                 start_date=None, cost: float = TRANSACTION_COST) -> dict:
    """
    Vektörel backtest motoru.
    - Sinyal 1  → uzun pozisyon (bir sonraki barda giriş)
    - Sinyal -1 → kısa pozisyon (BIST'te CFD/Vadeli ile uygulanabilir; spot için 0 al)
    - Sinyal 0  → nakit
    """
    data = df.copy()
    data['Signal'] = signals

    # Test dönemine kırp
    if start_date is not None:
        data = data[data.index >= start_date]

    data = data.dropna(subset=['Log_Return'])

    # Bir sonraki barda işlem yap (look-ahead bias önlemi)
    data['Position'] = data['Signal'].shift(1).fillna(0).clip(-1, 1)

    # Ham getiri (pozisyon × bar getirisi)
    data['Strat_Return'] = data['Position'] * data['Log_Return']

    # İşlem maliyeti: pozisyon değiştiğinde uygula
    pos_change = data['Position'].diff().abs()
    data['Cost'] = pos_change * cost
    data['Net_Return'] = data['Strat_Return'] - data['Cost']

    # Kümülatif getiri
    data['Cum_Return']  = np.exp(data['Net_Return'].cumsum())
    data['BH_Return']   = np.exp(data['Log_Return'].cumsum())   # Buy & Hold kıyası

    # ── Metrikler ─────────────────────────────────────────────────
    total_ret = data['Cum_Return'].iloc[-1] - 1

    # Annualize faktörü (bar başına)
    bars_per_year = {'1d': 252, '1h': 252*7, '15m': 252*28, '5m': 252*78}.get(INTERVAL, 252)
    mean_ret = data['Net_Return'].mean()
    std_ret  = data['Net_Return'].std() + 1e-9
    sharpe   = (mean_ret / std_ret) * np.sqrt(bars_per_year)

    # Max Drawdown
    roll_max = data['Cum_Return'].cummax()
    drawdown = data['Cum_Return'] / roll_max - 1
    max_dd   = drawdown.min()

    # Calmar
    calmar = (total_ret / abs(max_dd)) if max_dd != 0 else np.nan

    # Win Rate: pozisyon açık barlarda kazanma yüzdesi
    active   = data[data['Position'] != 0]
    win_rate = (active['Net_Return'] > 0).mean() * 100 if len(active) > 0 else np.nan

    # İşlem sayısı
    trades = int(pos_change.sum() / 2)

    # Buy & Hold getirisi (kıyas)
    bh_ret = data['BH_Return'].iloc[-1] - 1

    return {
        'Strateji'       : strategy_name,
        'Toplam Getiri %': round(total_ret * 100, 2),
        'BH Getiri %'    : round(bh_ret * 100, 2),
        'Sharpe'         : round(sharpe, 3),
        'Max Drawdown %' : round(max_dd * 100, 2),
        'Calmar'         : round(calmar, 3) if not np.isnan(calmar) else 'N/A',
        'Win Rate %'     : round(win_rate, 2),
        'İşlem Sayısı'   : trades,
        '_data'          : data,   # görselleştirme için
    }


print('Backtest motoru tanımlandı ✓')

In [ ]:
# Tüm stratejileri test döneminde backtest et
res_A = run_backtest(df, sig_A, 'A: Klasik TA',   start_date=test_start)
res_B = run_backtest(df, sig_B, 'B: GARCH',       start_date=test_start)
res_C = run_backtest(df, sig_C, 'C: XGBoost',     start_date=test_start)

results = [res_A, res_B, res_C]

# Kıyaslama tablosu
metric_cols = ['Strateji','Toplam Getiri %','BH Getiri %','Sharpe',
               'Max Drawdown %','Calmar','Win Rate %','İşlem Sayısı']
comparison_df = pd.DataFrame(results)[metric_cols]

print('\n' + '═'*80)
print('PERFORMANS KIYASLAMA TABLOSU')
print('═'*80)
print(comparison_df.to_string(index=False))
print('═'*80)

## 🏆 6. ŞAMPİYON STRATEJİ İLANI

In [ ]:
def declare_champion(results: list) -> dict:
    """Bileşik skor ile şampiyon stratejiyi belirle."""
    scores = []
    for r in results:
        # Normalize edilmiş bileşik skor (her metrik eşit ağırlıklı)
        ret_score    = r['Toplam Getiri %']
        sharpe_score = r['Sharpe'] * 20          # ölçek normalize
        dd_score     = -r['Max Drawdown %']      # pozitife çevir
        wr_score     = r['Win Rate %'] if not pd.isna(r['Win Rate %']) else 0
        composite    = 0.35*ret_score + 0.30*sharpe_score + 0.20*dd_score + 0.15*wr_score
        scores.append((r['Strateji'], composite, r))

    champion = max(scores, key=lambda x: x[1])

    print('\n' + '🏆'*20)
    print(f'  ŞAMPİYON STRATEJİ: {champion[0]}')
    print('🏆'*20)
    r = champion[2]
    print(f"  Toplam Getiri : {r['Toplam Getiri %']:+.2f}%")
    print(f"  Sharpe Oranı  : {r['Sharpe']:.3f}")
    print(f"  Max Drawdown  : {r['Max Drawdown %']:.2f}%")
    print(f"  Win Rate      : {r['Win Rate %']:.2f}%")
    print(f"  İşlem Sayısı  : {r['İşlem Sayısı']}")
    print(f"  Bileşik Skor  : {champion[1]:.2f}")

    # Sıralama
    print('\n  SIRALAMA:')
    for rank, (name, score, _) in enumerate(sorted(scores, key=lambda x: x[1], reverse=True), 1):
        print(f'    {rank}. {name}  (skor: {score:.2f})')

    return champion[2]


champion = declare_champion(results)

## 📈 7. GÖRSELLEŞTİRME

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), facecolor='#0d1117')
fig.suptitle(f'{TICKER} — Strateji Karşılaştırması (Test Dönemi)', 
             fontsize=16, color='white', fontweight='bold', y=0.98)

# Panel 1: Kümülatif Getiri
ax1 = axes[0]
ax1.set_facecolor('#0d1117')
for i, r in enumerate(results):
    cum = r['_data']['Cum_Return']
    ax1.plot(cum.index, cum.values, label=f"{r['Strateji']} ({r['Toplam Getiri %']:+.1f}%)",
             color=COLORS[i], linewidth=1.8)
# Buy & Hold
bh = results[0]['_data']['BH_Return']
ax1.plot(bh.index, bh.values, '--', color='gray', alpha=0.6,
         label=f"Buy & Hold ({results[0]['BH Getiri %']:+.1f}%)")
ax1.set_ylabel('Kümülatif Getiri (1 = başlangıç)', color='white')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.15)
ax1.tick_params(colors='white')
ax1.spines[['top','right','left','bottom']].set_color('#333')

# Panel 2: Drawdown
ax2 = axes[1]
ax2.set_facecolor('#0d1117')
for i, r in enumerate(results):
    cum = r['_data']['Cum_Return']
    roll_max = cum.cummax()
    dd = (cum / roll_max - 1) * 100
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.4, color=COLORS[i],
                     label=f"{r['Strateji']} (MaxDD: {r['Max Drawdown %']:.1f}%)")
ax2.set_ylabel('Drawdown (%)', color='white')
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(alpha=0.15)
ax2.tick_params(colors='white')
ax2.spines[['top','right','left','bottom']].set_color('#333')

# Panel 3: Fiyat + Sinyaller (XGBoost)
ax3 = axes[2]
ax3.set_facecolor('#0d1117')
price_test = df.loc[df.index >= test_start, 'Close']
ax3.plot(price_test.index, price_test.values, color='#aaaaaa', linewidth=1, label='Fiyat')

# Al sinyalleri
buy_mask  = (sig_C == 1)  & (df.index >= test_start)
sell_mask = (sig_C == -1) & (df.index >= test_start)
ax3.scatter(df.index[buy_mask],  df.loc[buy_mask, 'Close'],
            marker='^', color='#00ff88', s=40, label='XGBoost Al', zorder=5, alpha=0.8)
ax3.scatter(df.index[sell_mask], df.loc[sell_mask, 'Close'],
            marker='v', color='#ff6b6b', s=40, label='XGBoost Sat', zorder=5, alpha=0.8)
ax3.set_ylabel('Fiyat (TL)', color='white')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(alpha=0.15)
ax3.tick_params(colors='white')
ax3.spines[['top','right','left','bottom']].set_color('#333')

for ax in axes:
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_color('white')

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Grafik kaydedildi: strategy_comparison.png')

In [ ]:
# XGBoost Özellik Önem Grafiği
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0d1117')

# Sol: Feature importance
top20 = feat_imp.head(20)
bars  = ax1.barh(top20.index[::-1], top20.values[::-1], color='#4ecdc4', alpha=0.8)
ax1.set_xlabel('Önem Skoru', color='white')
ax1.set_title('XGBoost — Top 20 Özellik', color='white', fontweight='bold')
ax1.set_facecolor('#0d1117')
ax1.tick_params(colors='white')
ax1.spines[['top','right','left','bottom']].set_color('#333')
for label in ax1.get_xticklabels() + ax1.get_yticklabels():
    label.set_color('white')

# Sağ: Metrik karşılaştırma (radar benzeri bar)
metrics  = ['Toplam Getiri %', 'Sharpe', 'Win Rate %']
names    = [r['Strateji'].split(':')[0] for r in results]
x        = np.arange(len(metrics))
width    = 0.25

for i, r in enumerate(results):
    vals = [r['Toplam Getiri %'], r['Sharpe']*10, r['Win Rate %']]
    ax2.bar(x + i*width, vals, width, label=r['Strateji'], color=COLORS[i], alpha=0.8)

ax2.set_xticks(x + width)
ax2.set_xticklabels(['Getiri %', 'Sharpe×10', 'Win Rate %'], color='white')
ax2.set_title('Metrik Karşılaştırması', color='white', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_facecolor('#0d1117')
ax2.tick_params(colors='white')
ax2.spines[['top','right','left','bottom']].set_color('#333')
ax2.axhline(0, color='white', linewidth=0.5)
for label in ax2.get_xticklabels() + ax2.get_yticklabels():
    label.set_color('white')

plt.suptitle(f'{TICKER} — Model Analizi', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## ⚡ 8. CANLI SİNYAL ÜRETİCİ — Şu Anki Bar İçin Tahmin

In [ ]:
def generate_live_signal(ticker: str, index_ticker: str, interval: str,
                         model, scaler, feature_cols: list) -> dict:
    """
    Gerçek zamanlı (veya en güncel) bar için tüm stratejilerin sinyalini üret.
    """
    print(f'\n{"═"*60}')
    print(f'CANLI SİNYAL — {ticker} | {interval} | {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print('═'*60)

    # En son veriyi indir (kısa periyot)
    live_period = {'1d': '1y', '1h': '60d', '15m': '30d', '5m': '10d'}.get(interval, '60d')
    live_df = download_data(ticker, index_ticker, interval, live_period)
    live_df = build_features(live_df)

    last = live_df.dropna().iloc[-1]
    prev = live_df.dropna().iloc[-2]

    # ── Strateji A Sinyali ──────────────────────────────────────
    macd_cross = (last['MACD_Hist'] > 0) and (prev['MACD_Hist'] <= 0)
    rsi_ok     = 30 < last['RSI'] < 70
    trend_up   = last['EMA9'] > last['EMA21']
    vol_up     = last['Vol_Ratio'] > 1.0

    if macd_cross and rsi_ok and trend_up and vol_up:
        sig_a = 'AL  ▲'
    elif (last['MACD_Hist'] < 0 and prev['MACD_Hist'] >= 0) or (last['EMA9'] < last['EMA21']):
        sig_a = 'SAT ▼'
    else:
        sig_a = 'BEKLE ─'

    # ── Strateji C (XGBoost) Sinyali ───────────────────────────
    avail = [c for c in feature_cols if c in live_df.columns]
    last_features = live_df[avail].dropna().iloc[-1:]
    if len(last_features) > 0:
        last_sc  = scaler.transform(last_features)
        pred     = model.predict(last_sc)[0]
        proba    = model.predict_proba(last_sc)[0]
        sig_c    = f'AL  ▲ (güven: {proba[1]:.1%})' if pred == 1 else f'SAT ▼ (güven: {proba[0]:.1%})'
    else:
        sig_c = 'Yeterli veri yok'

    # ── Özet Çıktı ─────────────────────────────────────────────
    print(f'\nFiyat       : {last["Close"]:.2f} TL')
    print(f'RSI-14      : {last["RSI"]:.1f}')
    print(f'MACD Hist   : {last["MACD_Hist"]:.4f}')
    print(f'EMA9/21     : {last["EMA9"]:.2f} / {last["EMA21"]:.2f}')
    print(f'ATR %       : {last["ATR_Pct"]:.3%}')
    print(f'Hacim Oranı : {last["Vol_Ratio"]:.2f}x')
    print(f'ADX         : {last["ADX"]:.1f}')
    print(f'BB Pozisyon : {last["BB_Pct"]:.1%}')
    print()
    print(f'Strateji A (Klasik TA) : {sig_a}')
    print(f'Strateji C (XGBoost)   : {sig_c}')

    # Scalp önerileri
    atr     = last['ATR']
    close   = last['Close']
    stop    = round(close - 1.5 * atr, 2)
    target  = round(close + 2.5 * atr, 2)
    rr      = round((target - close) / (close - stop), 2) if close > stop else 'N/A'
    print(f'\n[SCALP PARAMETRELERİ]')
    print(f'  Giriş       : {close:.2f} TL')
    print(f'  Stop Loss   : {stop:.2f} TL  (-{(close-stop)/close:.1%})')
    print(f'  Hedef       : {target:.2f} TL  (+{(target-close)/close:.1%})')
    print(f'  Risk/Ödül   : 1:{rr}')

    return {'sig_a': sig_a, 'sig_c': sig_c, 'close': close,
            'stop': stop, 'target': target, 'rr': rr}


# Canlı sinyal üret
live = generate_live_signal(TICKER, INDEX_TKR, INTERVAL, xgb_model, xgb_scaler, FEATURE_COLS)

## 🔍 9. BIST-100 TARAYICI — En İyi Fırsatları Bul

In [ ]:
# BIST-100'ün popüler hisseleri (genişletilebilir)
BIST100_WATCHLIST = [
    'THYAO.IS', 'SISE.IS', 'EREGL.IS', 'AKBNK.IS', 'GARAN.IS',
    'KCHOL.IS', 'KOZAL.IS', 'BIMAS.IS', 'FROTO.IS', 'TUPRS.IS',
    'ASELS.IS', 'TOASO.IS', 'SAHOL.IS', 'YKBNK.IS', 'HALKB.IS',
    'PGSUS.IS', 'TAVHL.IS', 'EKGYO.IS', 'DOHOL.IS', 'SOKM.IS',
]

def scan_bist(watchlist: list, interval: str = '1d', period: str = '1y',
              top_n: int = 5) -> pd.DataFrame:
    """Tüm listedeki hisseler için hızlı teknik tarama yap."""
    records = []
    for tkr in watchlist:
        try:
            raw = yf.download(tkr, period=period, interval=interval,
                              progress=False, auto_adjust=True)
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)
            if raw.empty or len(raw) < 60:
                continue
            raw.index = pd.to_datetime(raw.index)
            if raw.index.tz is not None:
                raw.index = raw.index.tz_localize(None)
            raw['Endeks_Close'] = raw['Close']

            d = build_features(raw)
            last = d.dropna().iloc[-1]

            # Kalite puanı: momentum + düşük oynaklık + hacim baskısı
            score = 0
            score += 1 if last['RSI'] > 50 else -1
            score += 1 if last['MACD_Hist'] > 0 else -1
            score += 1 if last['EMA9'] > last['EMA21'] else -1
            score += 1 if last['Vol_Ratio'] > 1.2 else 0
            score += 1 if last['ADX'] > 25 else 0
            score += 1 if last['ROC5'] > 0 else -1
            score += 1 if last['BB_Pct'] > 0.5 else 0

            records.append({
                'Hisse'       : tkr,
                'Fiyat'       : round(last['Close'], 2),
                'RSI'         : round(last['RSI'], 1),
                'MACD_Hist'   : round(last['MACD_Hist'], 4),
                'ATR_Pct'     : round(last['ATR_Pct'] * 100, 2),
                'Vol_Ratio'   : round(last['Vol_Ratio'], 2),
                'ADX'         : round(last['ADX'], 1),
                'ROC5'        : round(last['ROC5'], 2),
                'Skor'        : score,
            })
        except Exception as e:
            pass

    df_scan = pd.DataFrame(records).sort_values('Skor', ascending=False)
    return df_scan


print('BIST Tarayıcı çalışıyor...')
scan_results = scan_bist(BIST100_WATCHLIST, interval='1d', period='1y', top_n=5)

print('\n' + '═'*70)
print('BIST-100 GÜÇLÜ SİNYAL TARAMASI (Bugün)')
print('═'*70)
print(scan_results.to_string(index=False))
print('\n⭐ En yüksek skorlu hisseler scalp fırsatı olarak değerlendirilebilir.')

## 🔎 10. BENZERLİK PATERNİ TARAYICISI (DTW Tabanlı)

In [ ]:
from scipy.spatial.distance import euclidean

def find_similar_patterns(df: pd.DataFrame, lookback: int = 20,
                           n_matches: int = 5) -> pd.DataFrame:
    """
    Son 'lookback' barı referans al, geçmişte en benzer desenleri bul.
    Normalize edilmiş Öklid mesafesi kullanır (hızlı DTW yerine).
    Eşleşme sonrası getiriyi döndürür → istatistiksel ön görü.
    """
    prices = df['Close'].values
    rets   = df['Log_Return'].dropna().values

    # Referans desen: son lookback barın normalize getirileri
    ref_window = rets[-lookback:]
    ref_norm   = (ref_window - ref_window.mean()) / (ref_window.std() + 1e-9)

    matches = []
    horizon = 5   # ileriye kaç bar bak

    for i in range(lookback, len(rets) - lookback - horizon):
        window = rets[i-lookback:i]
        norm   = (window - window.mean()) / (window.std() + 1e-9)
        dist   = np.linalg.norm(ref_norm - norm)   # L2
        future_ret = np.sum(rets[i:i+horizon])      # sonraki horizon bar getirisi
        matches.append({'idx': i, 'dist': dist,
                        'date': df.index[i].date(),
                        f'Sonraki {horizon} bar': round(future_ret * 100, 2)})

    res = pd.DataFrame(matches).sort_values('dist').head(n_matches)
    res = res[['date', 'dist', f'Sonraki {horizon} bar']]
    res.columns = ['Tarih', 'Mesafe', f'Sonraki {horizon}bar Getiri %']

    avg_fwd = res[f'Sonraki {horizon}bar Getiri %'].mean()
    direction = 'YUKARI ▲' if avg_fwd > 0 else 'AŞAĞI ▼'

    print(f'\n[PATERİN ANALİZİ] Son {lookback} bar ile en benzer {n_matches} geçmiş desen:')
    print(res.to_string(index=False))
    print(f'\n→ Ortalama sonraki {horizon} bar getiri tahmini: {avg_fwd:+.2f}% ({direction})')

    return res


pattern_res = find_similar_patterns(df.dropna(), lookback=20, n_matches=5)

## 📋 11. ÖZET RAPOR

In [ ]:
print('\n' + '╔' + '═'*62 + '╗')
print('║' + '  BIST SCALP TRADİNG — ÖZET RAPOR'.center(62) + '║')
print('╠' + '═'*62 + '╣')
print(f'║  Hisse      : {TICKER:<46}║')
print(f'║  Zaman Dil. : {INTERVAL:<46}║')
print(f'║  Test Dönem : {str(test_start.date()):<46}║')
print('╠' + '═'*62 + '╣')

for r in results:
    line = f"  {r['Strateji']:<20} | Getiri: {r['Toplam Getiri %']:+6.1f}% | Sharpe: {r['Sharpe']:5.2f} | WR: {r['Win Rate %']:5.1f}%"
    print(f'║{line:<62}║')

print('╠' + '═'*62 + '╣')
champ_line = f"  ŞAMPİYON: {champion['Strateji']}"
print(f'║{champ_line:<62}║')
print('╚' + '═'*62 + '╝')

print()
print(comparison_df.to_string(index=False))